In [ ]:
# Insérez votre code ici
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
df = pd.read_csv("admissions.csv")
df.head()
# df.info()
# df.describe(s)

In [ ]:
# data preparation
df = df.dropna()
test_gre = pd.cut(x=df["gre"],
      bins= [200,450,550,620,800],
      labels=['bad','average','decent','good'])
df = df.merge(test_gre, left_index=True, right_index=True)
df.drop(columns='gre_x',axis=1,inplace=True)
df = df.rename({'gre_y' : 'gre'},axis=1)

test_gpa = pd.cut(x= df['gpa'],
                 bins=[2,2.8,3.2,3.6,4],
                 labels=['bad','average','decent','good'])
df = df.merge(test_gpa, left_index=True, right_index=True)
df = df.drop(columns='gpa_x', axis = 1)
df = df.rename({'gpa_y' : 'gpa'}, axis=1)
df.head(5)

In [ ]:
# Logistic Regression
data = df.drop('admit', axis=1)
target = df['admit']

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=66)
enc = OneHotEncoder(handle_unknown='ignore')
X_train_enc = enc.fit_transform(X_train)
X_test_enc = enc.transform(X_test)

clf = linear_model.LogisticRegression(C=1.0)
clf.fit(X_train_enc, y_train)

y_pred = clf.predict(X_test_enc)
cm = confusion_matrix(y_test,y_pred)
# or cm = pd.crosstab(y_test, y_pred, rownames=['Classe réelle'], colnames=['Classe prédite'])

clf.score(X_test_enc, y_test)
print(classification_report(y_test, y_pred))

# to use a custom probability threshold for logistic regression
probs = clf.predict_proba(X_test_enc)
y_preds = np.where(probs[:,1]>0.4,1,0)
cm = pd.crosstab(y_test, y_preds, rownames=['Classe réelle'], colnames=['Classe prédite'])

fpr, tpr, seuils = roc_curve(y_test, probs[:,1], pos_label=1)
roc_auc = auc(fpr, tpr)

In [ ]:
#plot the results

plt.plot(fpr, tpr, color='orange', lw=2, label='Modèle clf (auc = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Aléatoire (auc = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux faux positifs')
plt.ylabel('Taux vrais positifs')
plt.title('Courbe ROC')
plt.legend(loc="lower right")
plt.show();